In [ ]:
# Descubrimiento robusto de la raiz del proyecto (no usar ../ relativo)
import pathlib as _pl, os as _os, sys as _sys
_ROOT = _pl.Path().resolve()
while not (_ROOT / "pyproject.toml").exists() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
_os.chdir(_ROOT)
if str(_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_ROOT))
if str(_ROOT / "src") not in _sys.path:
    _sys.path.insert(0, str(_ROOT / "src"))
print("cwd:", _os.getcwd())

# --- MLflow live (mlflow-live-notebooks): env BEFORE any TF/Keras import ---
_os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
_os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")
_os.environ.setdefault("TF_NUM_INTEROP_THREADS", "1")
_os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "1")
_os.environ.setdefault("OMP_NUM_THREADS", "1")
from ds02_rul.mlflow_live import log_live_run, persist_timings, setup_mlflow
setup_mlflow()
print("mlflow live: store=mlruns-live experiment=ds02-clase (fuente=live_notebook)")


# 00 · Datos de clase — `raw` → `data/processed/`

Este cuaderno es el **primer eslabón** de la secuencia: construye los 8 parquet
y `auditoria_nulos.json` desde el HDF5 DS02. **No contiene la receta**: la
receta vive en `src/clase/dataset_clase.py` y este cuaderno solo la invoca.

**Pipeline (orden exacto):**

1. `build_clase_frame` — une dev + test.
2. `inyectar_nulos_mar` — MAR ~2 % en T30, solo crucero alto (`alt > 27_000 & Mach > 0.68`, seed 7).
3. `split_por_unidad` — train/test por motor (11/14/15 fuera de train).
4. `imputar_ffill` — causal por unidad + mediana **solo de train**.
5. `asignar_fase_vuelo` — Ascenso / Crucero / Descenso desde `alt` y `Mach`.

**Comando canónico:** `python -m src.clase.dataset_clase`

**Verificar sin escribir:** `python -m src.clase.dataset_clase --check`

> `data/processed/` es *build output* y no se versiona. El builder es idempotente:
> si el contenido no cambió, no reescribe.


## Para qué se construyó este dataset

El objetivo es estimar la vida útil restante (RUL) de un motor turbofán
DS02 desde su telemetría: cuántos ciclos le quedan antes de necesitar
mantenimiento. Ese número alimenta una decisión de hangar. El motor sigue
volando (APTO), pide una inspección programada (INSPECCION) o entra a
hangar (HANGAR). Si las reglas no pueden decidir con lo que tienen, el
estado es SIN_DECISION y no se notifica nada.

El modelo desplegado es el bundle C: BiLSTM + LightGBM cuantílico, meta
RidgeCV/LightGBM, EMA causal 5 y banda conformal asimétrica al 90 %. Sobre
el manifiesto, con seed 42, RMSE 4.338 ciclos, MAE 3.227 y R² 0.9461 en
las 6.196 ventanas de test. Las reglas usan dos umbrales: 20 ciclos para
hangar y 25 para APTO (ADR-013).

Los parquet que salen de aquí son la entrada de todo lo demás. Por eso el
split y la imputación pesan tanto como la arquitectura:

- La unidad es el grupo. Un motor nunca cae en train y en test a la vez.
  Si se mezclan, el modelo aprende a reconocer el motor y la métrica deja
  de medir lo que dice medir.
- El test no se toca. Las unidades 11, 14 y 15 quedan fuera del
  entrenamiento y se evalúan una sola vez.
- La imputación es causal. `ffill` por unidad y mediana calculada solo con
  train, para que el relleno no transporte información del futuro.

El objetivo completo, el EDA y el contrato de columnas están en
`01_eda_y_contexto.ipynb`. El modelo y su evaluación, en los cuadernos 04
a 07. Aquí solo se dejan los parquet listos.

In [1]:
# Bootstrap: raíz del repo y src/ en sys.path (mismo patrón que NB01-NB07)
import pathlib as _pl, os as _os, sys as _sys
_ROOT = _pl.Path.cwd().resolve()
while not (_ROOT / "pyproject.toml").exists() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
_os.chdir(_ROOT)
for _p in (str(_ROOT), str(_ROOT / "src")):
    if _p not in _sys.path:
        _sys.path.insert(0, _p)
_os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

from src.clase.dataset_clase import construir_artefactos_clase

res = construir_artefactos_clase()

print("filas:", res["filas"])
print("unidades:", res["unidades"])
audit = res["auditoria"]
print("MAR:", audit["mecanismo"], "| nulos:", audit["nulos_inyectados"],
      "| elegibles:", audit["filas_elegibles"],
      "| tasa global T30:", round(audit["tasa_global_T30"], 4))
print("rachas >3:", audit["unidades_con_mas_de_3_seguidos"])
print("escritos:", res["escritos"] or "ninguno (ya estaba al día)")

filas: {'train_con_nulos.parquet': 52635, 'test_con_nulos.parquet': 12538, 'train.parquet': 52635, 'test.parquet': 12538, 'X_train.parquet': 52635, 'X_test.parquet': 12538, 'y_train.parquet': 52635, 'y_test.parquet': 12538}
unidades: {'train': [2.0, 5.0, 10.0, 16.0, 18.0, 20.0], 'test': [11.0, 14.0, 15.0]}
MAR: MAR-altitud | nulos: 1185 | elegibles: 12499 | tasa global T30: 0.0182
rachas >3: 0
escritos: ninguno (ya estaba al día)


**Qué mirar.** `escritos: ninguno` significa que los artefactos ya existían y
el contenido coincide (idempotencia). Los números de referencia son **52.635**
filas de train, **12.538** de test, **1.185** nulos MAR y **12.499** filas
elegibles.

**Siguiente:** `01_eda_y_contexto.ipynb`. Si cambias la inyección de nulos o los
umbrales, borra `data/processed/` y vuelve a ejecutar este cuaderno. Para
comprobar sin escribir nada: `python -m src.clase.dataset_clase --check`.


In [ ]:
# --- MLflow live: datos (params + counts; no training here) ---
from ds02_rul.mlflow_live import log_live_run, persist_timings
log_live_run(
    "live__nb00_datos",
    params={"notebook": "00_datos", "seed": 42},
    metrics={
        "n_train": float(res["filas"]["train"]),
        "n_test": float(res["filas"]["test"]),
    },
    notebook="00",
)
print("[NB00 MLflow live OK]")
